# 7.2 K-Means Clustering

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.02_k-means-clustering.ipynb)

Companion code for Section 7.2 of the book. Run the cells in order from top to bottom.

**Example 7.2.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_score, calinski_harabasz_score
import warnings; warnings.filterwarnings('ignore')

# ── Generate well-separated clusters ─────────────────────────────
np.random.seed(42)
X, y_true = make_blobs(n_samples=500, n_features=2, centers=4,
                         cluster_std=0.8, random_state=42)
X = StandardScaler().fit_transform(X)

# ── Fit K-Means with different k ─────────────────────────────────
k_range  = range(2, 11)
inertias = []
sil_scores = []
ch_scores  = []

for k in k_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    labels = km.labels_
    sil_scores.append(silhouette_score(X, labels))
    ch_scores.append(calinski_harabasz_score(X, labels))

print(f'{'k':>4} {'WCSS (Inertia)':>16} {'Silhouette':>12} {'Calinski-H':>12}')
print('='*50)
for k, wc, sl, ch in zip(k_range, inertias, sil_scores, ch_scores):
    marker = ' ← optimal' if k==4 else ''
    print(f'{k:>4} {wc:>16.2f} {sl:>12.4f} {ch:>12.2f}{marker}')

# ── Fit the best model ───────────────────────────────────────────
best_k = 4
km_best = KMeans(n_clusters=best_k, init='k-means++', n_init=20, random_state=42)
km_best.fit(X)
labels   = km_best.labels_
centers  = km_best.cluster_centers_

print(f'\nK-Means (k={best_k}) Results:')
print(f'  Inertia (WCSS):    {km_best.inertia_:.4f}')
print(f'  Silhouette Score:  {silhouette_score(X, labels):.4f}')
print(f'  Iterations:        {km_best.n_iter_}')
print(f'\nCluster sizes: {dict(zip(*np.unique(labels, return_counts=True)))}')
print(f'\nCluster centers:')
for i, c in enumerate(centers):
    print(f'  Cluster {i}: [{c[0]:+.4f}, {c[1]:+.4f}]')

# ── Convergence comparison: random vs K-Means++ ──────────────────
print('\nInitialisation comparison (k=4, 20 runs each):')
for init_method in ['random', 'k-means++']:
    km_test = KMeans(n_clusters=4, init=init_method, n_init=20, random_state=42)
    km_test.fit(X)
    print(f'  {init_method:12s}: WCSS={km_test.inertia_:.4f}  Iterations={km_test.n_iter_}')

# ── Comprehensive visualisation ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('K-Means Clustering Analysis', fontsize=14, fontweight='bold')
sns.set_theme(style='whitegrid')

palette = ['#E53935','#1976D2','#43A047','#FF6F00']

# 1. Raw data
ax = axes[0,0]
ax.scatter(X[:,0], X[:,1], s=15, alpha=0.7, color='gray')
ax.set_title('Raw Data (No Labels)'); ax.set_xlabel('Feature 1'); ax.set_ylabel('Feature 2')

# 2. K-Means result
ax = axes[0,1]
for k in range(best_k):
    mask = labels == k
    ax.scatter(X[mask,0], X[mask,1], s=15, alpha=0.7, color=palette[k], label=f'Cluster {k}')
ax.scatter(centers[:,0], centers[:,1], s=300, c='black', marker='*', zorder=10,
          edgecolors='white', linewidth=1.5, label='Centroids')
ax.set_title(f'K-Means (k={best_k})'); ax.set_xlabel('Feature 1')
ax.legend(fontsize=8);

# 3. True labels comparison
ax = axes[0,2]
for k in range(4):
    mask = y_true == k
    ax.scatter(X[mask,0], X[mask,1], s=15, alpha=0.7, color=palette[k])
ax.set_title('True Labels (Ground Truth)'); ax.set_xlabel('Feature 1')

# 4. Elbow curve
ax = axes[1,0]
ax.plot(k_range, inertias, 'bo-', ms=8, lw=2)
ax.axvline(x=4, color='red', ls='--', lw=2, label='Optimal k=4')
ax.set_xlabel('Number of Clusters (k)'); ax.set_ylabel('WCSS (Inertia)')
ax.set_title('Elbow Method - Finding Optimal k'); ax.legend(); ax.grid(alpha=0.3)

# 5. Silhouette score
ax = axes[1,1]
ax.plot(k_range, sil_scores, 'rs-', ms=8, lw=2)
ax.axvline(x=4, color='red', ls='--', lw=2, label=f'Best k=4 ({max(sil_scores):.3f})')
ax.set_xlabel('Number of Clusters (k)'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score vs k\n(Higher = better defined clusters)')
ax.legend(); ax.grid(alpha=0.3)

# 6. Decision regions (Voronoi-like)
ax = axes[1,2]
h = 0.05
x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min,x_max,h), np.arange(y_min,y_max,h))
Z = km_best.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
for k in range(best_k):
    mask = labels == k
    ax.scatter(X[mask,0], X[mask,1], s=15, alpha=0.8, color=palette[k])
ax.scatter(centers[:,0], centers[:,1], s=300, c='black', marker='*', zorder=10)
ax.set_title('Voronoi Decision Regions'); ax.set_xlabel('Feature 1')

plt.tight_layout()
plt.savefig('kmeans_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Mini-Batch K-Means for large datasets ────────────────────────
print('\nMini-Batch K-Means (faster for large datasets):')
X_large, _ = make_blobs(n_samples=100000, centers=4, random_state=42)
import time
for cls, name in [(KMeans,'K-Means'),(MiniBatchKMeans,'MiniBatch K-Means')]:
    km_cls = cls(n_clusters=4, random_state=42)
    t0 = time.time()
    km_cls.fit(X_large)
    t1 = time.time()
    print(f'  {name:<22}: {t1-t0:.3f}s  inertia={km_cls.inertia_:,.0f}')